ЛР2 Сервер статистики

In [ ]:
from pathlib import Path
from xmlrpc.server import SimpleXMLRPCRequestHandler, SimpleXMLRPCServer
import csv
import datetime


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


project_directory = Path.cwd()
log_file = project_directory / 'events_log.csv'
maximum_log_records = 100


# ЛР2, статистика: хранить журнал событий в CSV с заголовком столбцов.
def create_log_file():
    with log_file.open('w', encoding='utf-8', newline='') as target_file:
        csv_writer = csv.writer(target_file)
        csv_writer.writerow(['timestamp', 'event_type', 'details'])


def read_log_file(source_log_file):
    if not source_log_file.exists():
        return []
    with source_log_file.open(encoding='utf-8', newline='') as source_file:
        return list(csv.DictReader(source_file))


# Уточнение для сдачи: фильтровать записи во всех журналах, включая архивные.
def get_log_files():
    # Для выборки используются все архивы и текущий журнал.
    all_log_files = sorted(project_directory.glob('events_log_*.csv'))
    if log_file.exists():
        all_log_files.append(log_file)
    return all_log_files


# ЛР2, ограничение журнала: перед следующей записью архивировать заполненный файл.
# Имя содержит дату и время YYYYMMDD_hhmmss, затем создаётся новый журнал.
def rotate_log():
    # При достижении лимита текущий журнал становится архивным.
    if len(read_log_file(log_file)) < maximum_log_records:
        return
    archive_timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    archived_log_file = project_directory / f'events_log_{archive_timestamp}.csv'
    # Суффикс нужен, если две ротации произошли в одну секунду.
    archive_number = 1
    while archived_log_file.exists():
        archived_log_file = project_directory / f'events_log_{archive_timestamp}_{archive_number}.csv'
        archive_number += 1
    log_file.rename(archived_log_file)
    create_log_file()


# ЛР2, статистика: принимать события основного сервера, сохранять время и тип операции.
# details дополнительно хранит результат или параметры операции.
def add_event(event_type, details=''):
    if not log_file.exists():
        create_log_file()
    rotate_log()
    event_record = {
        'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'event_type': str(event_type),
        'details': str(details),
    }
    with log_file.open('a', encoding='utf-8', newline='') as target_file:
        csv_writer = csv.DictWriter(target_file, fieldnames=event_record.keys())
        csv_writer.writerow(event_record)
    return True


# ЛР2: дать возможность настраивать максимальное количество записей журнала.
def set_max_log_records(record_limit):
    global maximum_log_records
    maximum_log_records = int(record_limit)
    if maximum_log_records < 1:
        maximum_log_records = 1
    return maximum_log_records


# ЛР2, клиентская выборка: вернуть журнал или срез по типу события и времени.
# Пустые параметры отключают фильтры; границы времени включаются в выборку.
def get_events(event_type='', start_time='', end_time=''):
    selected_events = []
    # Единый формат времени позволяет сравнивать даты как строки.
    for source_log_file in get_log_files():
        for event_record in read_log_file(source_log_file):
            if event_type and event_record['event_type'] != event_type:
                continue
            if start_time and event_record['timestamp'] < start_time:
                continue
            if end_time and event_record['timestamp'] > end_time:
                continue
            # Имя файла показывает, из текущего журнала или архива взята запись.
            event_record['source_log'] = source_log_file.name
            selected_events.append(event_record)
    return sorted(selected_events, key=lambda event_record: event_record['timestamp'])


# ЛР2: отдельный XML-RPC сервис статистики со своим портом и методами.
statistics_server = SimpleXMLRPCServer(('127.0.0.1', 8018), requestHandler=RequestHandler, allow_none=True, logRequests=True)
statistics_server.register_introspection_functions()
statistics_server.register_function(add_event, 'add_event')
statistics_server.register_function(set_max_log_records, 'set_max_log_records')
statistics_server.register_function(get_events, 'get_events')

print('Сервер статистики слушает порт 8018')
statistics_server.serve_forever()


Сервер статистики слушает порт 8018
